In [1]:
import argparse
import os
import pickle
import time

from importlib import metadata
import torch
try:
    try:
        if metadata.version("rsl-rl"):
            raise ImportError
    except metadata.PackageNotFoundError:
        if metadata.version("rsl-rl-lib") != "3.1.1":  #2.2.4
            raise ImportError
except (metadata.PackageNotFoundError, ImportError) as e:
    raise ImportError("Please uninstall 'rsl_rl' and install 'rsl-rl-lib==2.2.4'.") from e
from rsl_rl.runners import OnPolicyRunner

In [2]:
from bp000_env_cnoid import BP000Env as RLEnv

In [3]:
# 任意設定項目
# exp_name = 'collision-walking-rand'  # ckpt = 4000
exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-13'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-9'  # min_ankle_height 弊害
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-8'  #  暫定１位
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-6'
# exp_name = 'friction-walking-terrain2-kp2000kd50-kpkdrand-5'
ckpt = 100

action_scale = 1.0 # 動作のスケールを調整

In [4]:
# 既存のセルを置き換え
import pandas as pd
import numpy as np

# データ収集用のリスト
# action_data = []
obs_data = []
torque_data = []
step_data = []

# CSVファイルの準備
csv_filename = f'obs_data/{exp_name}_step_data.csv'
os.makedirs('obs_data', exist_ok=True)

In [5]:
def _obs_vec(obs):
    # TensorDict or dict → 'policy' を優先
    if isinstance(obs, dict) or hasattr(obs, "get"):
        if "policy" in obs:
            obs = obs["policy"]
    if torch.is_tensor(obs):
        return obs.detach().cpu().numpy().ravel()
    return np.asarray(obs, dtype=np.float32).ravel()

In [6]:
## set robot path fix collisiton 
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))  # /userdir
robot_path = os.path.join(ROOT, "userdir", "humanoid_research_k", "robots", "kawada_base.simple_collision.urdf")

In [7]:
log_dir = f"logs/{exp_name}"
env_cfg, obs_cfg, reward_cfg, command_cfg, train_cfg = pickle.load(open(f"logs/{exp_name}/cfgs.pkl", "rb"))
reward_cfg["reward_scales"] = {}

In [8]:
## override
# env_cfg["episode_length_s"] = 20.0
# command_cfg["lin_vel_x_range"] = [0.5, 0.5]
# env_cfg['dt'] = 0.001
# env_cfg['substeps'] = 10
# env_cfg["kd"] = 50
env_cfg['base_roll_noise'] = [0,0]
env_cfg['base_pitch_noise'] = [0,0]
env_cfg['termination_if_roll_greater_than'] = 150
env_cfg['termination_if_pitch_greater_than'] = 150
env_cfg['rotorInertia'] = 0.1
env_cfg["base_init_pos"] = [0.0, 0.0, 0.64]

In [9]:
reward_cfg

{'tracking_sigma': 0.25,
 'base_height_target': 0.64,
 'feet_height_target': 0.075,
 'reward_scales': {}}

In [10]:
env_cfg



{'num_actions': 12,
 'default_joint_angles': {'R_HIP_Y': 0.0,
  'R_HIP_R': 0.0,
  'R_HIP_P': -0.8,
  'R_KNEE': 1.6,
  'R_ANKLE_P': -0.8,
  'R_ANKLE_R': 0.0,
  'L_HIP_Y': 0.0,
  'L_HIP_R': 0.0,
  'L_HIP_P': -0.8,
  'L_KNEE': 1.6,
  'L_ANKLE_P': -0.8,
  'L_ANKLE_R': 0.0},
 'joint_names': ['R_HIP_Y',
  'R_HIP_R',
  'R_HIP_P',
  'R_KNEE',
  'R_ANKLE_P',
  'R_ANKLE_R',
  'L_HIP_Y',
  'L_HIP_R',
  'L_HIP_P',
  'L_KNEE',
  'L_ANKLE_P',
  'L_ANKLE_R'],
 'kp': 2000.0,
 'kd': 50.0,
 'termination_if_roll_greater_than': 150,
 'termination_if_pitch_greater_than': 150,
 'base_init_pos': [0.0, 0.0, 0.64],
 'base_init_quat': [1.0, 0.0, 0.0, 0.0],
 'episode_length_s': 20.0,
 'resampling_time_s': 4.0,
 'action_scale': 1.0,
 'simulate_action_latency': True,
 'clip_actions': 100.0,
 'dt': 0.01,
 'substeps': 10,
 'rotorInertia': 0.1,
 'base_roll_noise': [0, 0],
 'base_pitch_noise': [0, 0],
 'domain_rand': {'friction': [0.4, 1.1],
  'restitution': [0.0, 0.2],
  'kp': [1800.0, 2200.0],
  'kd': [25.0, 100.0]}

In [11]:
env = RLEnv(
    num_envs=1,
    env_cfg=env_cfg,
    obs_cfg=obs_cfg,
    reward_cfg=reward_cfg,
    command_cfg=command_cfg,
    dt=env_cfg['dt'],
    substeps=env_cfg['substeps'],
    show_viewer=True,
    robot_urdf_path=robot_path,
)

In [12]:
runner = OnPolicyRunner(env, train_cfg, log_dir, device='cuda')
resume_path = os.path.join(log_dir, f"model_{ckpt}.pt")
runner.load(resume_path)
policy = runner.get_inference_policy(device='cuda')

obs, _ = env.reset()
cnt = 0

torques = env.sim.sbody.getTorques()

print("obs : ", obs["policy"])

# データを記録
step_data.append(cnt)
obs_data.append(_obs_vec(obs))
torque_data.append(torques.copy())

cnt += 1

--------------------------------------------------------------------------------
Resolved observation sets: 
	 policy :  ['policy']
	 critic :  ['policy']
--------------------------------------------------------------------------------
Actor MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=12, bias=True)
)
Critic MLP: MLP(
  (0): Linear(in_features=45, out_features=512, bias=True)
  (1): ELU(alpha=1.0)
  (2): Linear(in_features=512, out_features=256, bias=True)
  (3): ELU(alpha=1.0)
  (4): Linear(in_features=256, out_features=128, bias=True)
  (5): ELU(alpha=1.0)
  (6): Linear(in_features=128, out_features=1, bias=True)
)
obs :  tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        

In [13]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 1
Original actions :  tensor([[-0.0741,  0.5741, -0.2769,  0.0519, -0.5489, -0.0078, -0.0146, -0.5134,
          0.0265,  0.3250, -0.6370,  0.1857]], device='cuda:0')
Scaled actions :  tensor([[-0.0741,  0.5741, -0.2769,  0.0519, -0.5489, -0.0078, -0.0146, -0.5134,
          0.0265,  0.3250, -0.6370,  0.1857]], device='cuda:0')


/userdir/irsl_rl/rl_env_base.py:110: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  self.exact_actions = torch.tensor(actions, device=self.device, dtype=torch.float32) ## copy
/userdir/irsl_rl/rl_env_cnoid.py:103: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  self.dof_pos = torch.tensor([self.convAnglesToGenesis(sbody.angleVector())]).to(torch.float32).to(self.device)


obs :  tensor([[-1.0485e-18,  1.5649e-08, -7.0849e-19,  5.0711e-10,  2.0808e-20,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  1.4362e-08,
          3.0845e-08, -1.4168e-04,  3.6454e-04, -1.9103e-04, -3.4389e-08,
         -8.5621e-08, -4.7636e-08, -1.4180e-04,  3.6466e-04, -1.9103e-04,
          7.6633e-07,  7.1809e-07,  1.5422e-06, -7.0859e-03,  1.8227e-02,
         -9.5517e-03, -1.7194e-06, -4.2811e-06, -2.3818e-06, -7.0895e-03,
          1.8232e-02, -9.5524e-03,  3.8316e-05, -7.4062e-02,  5.7410e-01,
         -2.7693e-01,  5.1903e-02, -5.4886e-01, -7.8433e-03, -1.4606e-02,
         -5.1343e-01,  2.6511e-02,  3.2502e-01, -6.3700e-01,  1.8575e-01]],
       device='cuda:0')
torques: [ 3.99683909e-16 -9.12643102e-16  2.42782550e-06  7.51007967e-06
  1.66224901e-06  2.18425700e-16  2.17253195e-16 -6.61016595e-16
  2.42782550e-06  7.51007967e-06  1.66224901e-06 -7.93165041e-17]
データ収集: step 2


In [14]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 2
Original actions :  tensor([[ 0.0168,  1.0974, -0.0347,  0.3772, -0.6228, -0.1638, -0.2231, -1.2023,
          0.2999,  0.4715, -0.8260,  0.2216]], device='cuda:0')
Scaled actions :  tensor([[ 0.0168,  1.0974, -0.0347,  0.3772, -0.6228, -0.1638, -0.2231, -1.2023,
          0.2999,  0.4715, -0.8260,  0.2216]], device='cuda:0')
obs :  tensor([[ 2.2048e-02,  1.0199e-01,  2.4855e-01,  2.4244e-03, -4.5938e-04,
         -1.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00, -2.0723e-02,
          6.8430e-03, -1.8990e-02,  2.3253e-02, -9.5488e-02, -4.2724e-03,
         -2.8044e-03, -9.8019e-03, -1.0052e-02,  3.0901e-02, -1.0624e-01,
          5.2531e-02, -1.0965e-01,  5.6948e-02, -1.4597e-01,  1.5795e-01,
         -7.3133e-01, -1.0512e-02, -1.5234e-02, -8.3175e-02, -6.7506e-02,
          2.3369e-01, -9.7120e-01,  2.8810e-01,  1.6811e-02,  1.0974e+00,
         -3.4677e-02,  3.7717e-01, -6.2285e-01, -1.6378e-01, -2.2309e-01,
         -1.2023e+00,  2.9991e-01,  4.7149e-01, -8.2599e-01,  2.2

In [15]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 3
Original actions :  tensor([[ 0.1594,  1.2094, -0.0766,  0.2824,  0.0326, -0.1920, -0.0989, -1.0729,
          0.0771,  0.1118, -0.1764,  0.0392]], device='cuda:0')
Scaled actions :  tensor([[ 0.1594,  1.2094, -0.0766,  0.2824,  0.0326, -0.1920, -0.0989, -1.0729,
          0.0771,  0.1118, -0.1764,  0.0392]], device='cuda:0')
obs :  tensor([[ 6.7867e-04, -4.1982e-01,  4.0282e-01, -4.5198e-03, -1.0050e-03,
         -9.9999e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00, -2.3353e-02,
          2.4381e-02, -4.0439e-02,  7.7397e-02, -2.6700e-01, -5.1408e-02,
         -4.9474e-02, -3.1402e-02, -6.7182e-03,  9.3317e-02, -3.3538e-01,
          1.1637e-01,  4.2619e-02,  1.1530e-01, -6.4991e-02,  3.5590e-01,
         -7.7859e-01, -2.4761e-01, -3.3273e-01, -1.3691e-01,  8.8944e-02,
          3.7206e-01, -1.0728e+00,  2.3417e-01,  1.5938e-01,  1.2094e+00,
         -7.6586e-02,  2.8239e-01,  3.2571e-02, -1.9204e-01, -9.8925e-02,
         -1.0729e+00,  7.7117e-02,  1.1178e-01, -1.7641e-01,  3.9

In [16]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 4
Original actions :  tensor([[ 0.1131,  1.0676, -0.3487, -0.1029,  0.1497, -0.1828,  0.2289, -1.0174,
         -0.3962, -0.2106,  0.0357,  0.0764]], device='cuda:0')
Scaled actions :  tensor([[ 0.1131,  1.0676, -0.3487, -0.1029,  0.1497, -0.1828,  0.2289, -1.0174,
         -0.3962, -0.2106,  0.0357,  0.0764]], device='cuda:0')
obs :  tensor([[-0.0791, -0.4652,  0.0729, -0.0232,  0.0012, -0.9997,  1.0000,  0.0000,
          0.0000,  0.0213,  0.0574, -0.0536,  0.1497, -0.3160, -0.1066, -0.0891,
         -0.0662,  0.0214,  0.1491, -0.4436,  0.1037,  0.2819,  0.2093, -0.0595,
          0.3252,  0.1917, -0.1924, -0.0567, -0.2088,  0.1734,  0.1786, -0.1043,
         -0.1328,  0.1131,  1.0676, -0.3487, -0.1029,  0.1497, -0.1828,  0.2289,
         -1.0174, -0.3962, -0.2106,  0.0357,  0.0764]], device='cuda:0')
torques: [   2.99856391  200.           11.40891745  -56.47257173  200.
   22.87595791   49.84294603 -200.          -50.39661942 -200.
  200.            2.18989318]
データ収集: step 5


In [17]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 5
Original actions :  tensor([[-0.0610,  0.7917,  0.0694, -0.3530,  0.0168, -0.2258, -0.0021, -1.0043,
         -0.1619, -0.4532,  0.0803,  0.1498]], device='cuda:0')
Scaled actions :  tensor([[-0.0610,  0.7917,  0.0694, -0.3530,  0.0168, -0.2258, -0.0021, -1.0043,
         -0.1619, -0.4532,  0.0803,  0.1498]], device='cuda:0')
obs :  tensor([[-0.0769,  0.1880, -0.1809, -0.0275,  0.0042, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0708,  0.1091, -0.0879,  0.1965, -0.1906, -0.1359, -0.0464,
         -0.1147,  0.0362,  0.1680, -0.3582,  0.0943,  0.1486,  0.2988, -0.2561,
          0.1591,  0.7396, -0.1097,  0.4185, -0.2683,  0.0013,  0.0265,  0.7942,
         -0.0303, -0.0610,  0.7917,  0.0694, -0.3530,  0.0168, -0.2258, -0.0021,
         -1.0043, -0.1619, -0.4532,  0.0803,  0.1498]], device='cuda:0')
torques: [ -71.09074052  200.         -200.         -200.          -52.67826197
   16.54450463  192.15182455 -200.         -200.         -200.
   52.99007885   -5.08969001]
データ収集:

In [18]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 6
Original actions :  tensor([[ 0.0592,  0.5336,  0.4848,  0.0994, -0.4170, -0.2257, -0.6878, -0.8970,
          0.2177, -0.1170, -0.4436,  0.2146]], device='cuda:0')
Scaled actions :  tensor([[ 0.0592,  0.5336,  0.4848,  0.0994, -0.4170, -0.2257, -0.6878, -0.8970,
          0.2177, -0.1170, -0.4436,  0.2146]], device='cuda:0')
obs :  tensor([[ 0.0437,  0.2510, -0.0865, -0.0188,  0.0043, -0.9998,  1.0000,  0.0000,
          0.0000,  0.0474,  0.1779, -0.1116,  0.1932, -0.0742, -0.1711,  0.0041,
         -0.1809,  0.0350,  0.1549, -0.2017,  0.1110, -0.2411,  0.3738, -0.0032,
         -0.1568,  0.2348, -0.1276,  0.0506, -0.3757, -0.0258, -0.1429,  0.6233,
          0.0857,  0.0592,  0.5336,  0.4848,  0.0994, -0.4170, -0.2257, -0.6878,
         -0.8970,  0.2177, -0.1170, -0.4436,  0.2146]], device='cuda:0')
torques: [   8.36183525  200.          200.         -200.          -85.16318207
   19.33434284  -88.9279632  -200.         -200.         -200.
  -62.29070797   -7.70539507]
データ収集:

In [19]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 7
Original actions :  tensor([[ 0.6181,  0.6857, -0.1719,  0.4387, -0.2817, -0.2978, -0.5745, -0.2041,
         -0.0900,  0.1993, -0.3036,  0.1996]], device='cuda:0')
Scaled actions :  tensor([[ 0.6181,  0.6857, -0.1719,  0.4387, -0.2817, -0.2978, -0.5745, -0.2041,
         -0.0900,  0.1993, -0.3036,  0.1996]], device='cuda:0')
obs :  tensor([[ 0.0673, -0.3372, -0.0416, -0.0219,  0.0016, -0.9998,  1.0000,  0.0000,
          0.0000,  0.0190,  0.2535, -0.0803,  0.1521, -0.1307, -0.1904, -0.0325,
         -0.2637,  0.0631,  0.0927, -0.1845,  0.1451, -0.0325,  0.3870,  0.2609,
         -0.2204, -0.5914, -0.0787, -0.3742, -0.4359,  0.2335, -0.4149, -0.3533,
          0.1526,  0.6181,  0.6857, -0.1719,  0.4387, -0.2817, -0.2978, -0.5745,
         -0.2041, -0.0900,  0.1993, -0.3036,  0.1996]], device='cuda:0')
torques: [ 125.57062125  191.62204928  200.          105.36608862  -12.79370361
    7.89193423 -200.         -200.           93.97577987  -33.63230493
 -200.          -14.00472009

In [20]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 8
Original actions :  tensor([[ 0.4628, -0.2412, -0.2701,  0.2971,  0.1896, -0.2892, -0.2371,  0.1088,
         -0.0631, -0.0165, -0.0487,  0.2599]], device='cuda:0')
Scaled actions :  tensor([[ 0.4628, -0.2412, -0.2701,  0.2971,  0.1896, -0.2892, -0.2371,  0.1088,
         -0.0631, -0.0165, -0.0487,  0.2599]], device='cuda:0')
obs :  tensor([[-0.5841,  0.1032, -0.0258, -0.0254,  0.0133, -0.9996,  1.0000,  0.0000,
          0.0000,  0.0742,  0.3557, -0.0805,  0.1338, -0.1983, -0.2327, -0.1728,
         -0.3152,  0.0465,  0.0665, -0.2304,  0.1825,  0.5374,  0.6113, -0.2291,
          0.0085, -0.4072, -0.3180, -0.9723, -0.1059, -0.3406,  0.1426, -0.3361,
          0.2424,  0.4628, -0.2412, -0.2701,  0.2971,  0.1896, -0.2892, -0.2371,
          0.1088, -0.0631, -0.0165, -0.0487,  0.2599]], device='cuda:0')
torques: [ 200.          200.         -198.73631771  200.         -195.6612439
 -152.74529576 -200.          200.         -200.          200.
 -170.0164823    52.79245504]
データ収集: 

In [21]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 9
Original actions :  tensor([[-0.2004, -1.1835,  0.5250,  0.3432, -0.0117, -0.3421,  0.0777,  0.4938,
          0.6376, -0.1553,  0.0614,  0.2265]], device='cuda:0')
Scaled actions :  tensor([[-0.2004, -1.1835,  0.5250,  0.3432, -0.0117, -0.3421,  0.0777,  0.4938,
          0.6376, -0.1553,  0.0614,  0.2265]], device='cuda:0')
obs :  tensor([[-0.6459, -0.0338, -0.2115, -0.0223,  0.0378, -0.9990,  1.0000,  0.0000,
          0.0000,  0.2025,  0.4666, -0.1498,  0.1476, -0.1732, -0.2685, -0.3408,
         -0.3241, -0.0093,  0.0841, -0.2002,  0.2250,  0.6086,  0.5249, -0.3525,
          0.1070,  0.5578, -0.0449, -0.6005, -0.0047, -0.1535,  0.0635,  0.3439,
          0.1299, -0.2004, -1.1835,  0.5250,  0.3432, -0.0117, -0.3421,  0.0777,
          0.4938,  0.6376, -0.1553,  0.0614,  0.2265]], device='cuda:0')
torques: [ -84.04579599 -200.          103.33772369  200.          200.
    6.61389445  200.          200.           62.24840253 -200.
  -32.99008787  -59.88802745]
データ収集: step 10

In [22]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 10
Original actions :  tensor([[-0.4712, -1.4260,  0.0991,  0.5826, -0.4490, -0.4208, -0.1475,  0.9001,
         -0.3298, -0.2607, -0.0535, -0.0533]], device='cuda:0')
Scaled actions :  tensor([[-0.4712, -1.4260,  0.0991,  0.5826, -0.4490, -0.4208, -0.1475,  0.9001,
         -0.3298, -0.2607, -0.0535, -0.0533]], device='cuda:0')
obs :  tensor([[-6.7220e-01, -4.8401e-01, -2.3489e-01, -3.4563e-02,  6.4044e-02,
         -9.9735e-01,  1.0000e+00,  0.0000e+00,  0.0000e+00,  2.5819e-01,
          5.7582e-01, -1.7812e-01,  1.7432e-01, -7.7988e-02, -2.9733e-01,
         -4.0113e-01, -3.2365e-01,  3.6049e-04,  8.5615e-02, -7.1465e-02,
          2.4500e-01, -6.2213e-04,  5.6617e-01,  4.7838e-02,  1.5921e-01,
          5.1185e-01, -2.1597e-01, -5.4457e-02,  2.0932e-03,  2.1241e-01,
         -5.9200e-02,  8.7254e-01,  2.6545e-02, -4.7124e-01, -1.4260e+00,
          9.9078e-02,  5.8258e-01, -4.4897e-01, -4.2079e-01, -1.4753e-01,
          9.0006e-01, -3.2981e-01, -2.6068e-01, -5.3513e-02, -5.

In [23]:
with torch.no_grad():
    print("cnt :", cnt)   
    actions = policy(obs)

    # アクションに倍率を適用して動きを制限
    scaled_actions = actions * action_scale

    print("Original actions : ", actions)
    print("Scaled actions : ", scaled_actions)

    obs, rews, dones, infos = env.step(scaled_actions) # スケール済みアクションを使用
    print("obs : ", obs["policy"])
    torques = env.sim.sbody.getTorques()
    print("torques:", torques)
    
    # データを記録
    step_data.append(cnt)
    # action_data.append(actions.cpu().numpy().flatten())
    obs_data.append(_obs_vec(obs))
    torque_data.append(torques.copy())
    
    cnt += 1

print(f"データ収集: step {cnt}")
     


cnt : 11
Original actions :  tensor([[ 0.1136, -1.3762,  0.2876,  0.5824, -0.4137, -0.6036, -0.7880,  0.7908,
         -0.5607, -0.1789, -0.2274,  0.2189]], device='cuda:0')
Scaled actions :  tensor([[ 0.1136, -1.3762,  0.2876,  0.5824, -0.4137, -0.6036, -0.7880,  0.7908,
         -0.5607, -0.1789, -0.2274,  0.2189]], device='cuda:0')
obs :  tensor([[-0.4969, -0.2284, -0.6276, -0.0510,  0.0862, -0.9950,  1.0000,  0.0000,
          0.0000,  0.1882,  0.6817, -0.1207,  0.2172, -0.0842, -0.3424, -0.3763,
         -0.3135,  0.0172,  0.1054, -0.0237,  0.2018, -0.6244,  0.4758,  0.4658,
          0.2737, -0.4731, -0.1735,  0.2793,  0.0934,  0.0089,  0.1660, -0.1678,
         -0.4699,  0.1136, -1.3762,  0.2876,  0.5824, -0.4137, -0.6036, -0.7880,
          0.7908, -0.5607, -0.1789, -0.2274,  0.2189]], device='cuda:0')
torques: [-200.         -200.           72.0348208   200.         -200.
  121.24380099  200.          200.         -200.         -200.
  -71.39048226 -200.        ]
データ収集: step 1

In [25]:
# 既存のforループを置き換え
num_steps = 10
for i in range(num_steps):
    with torch.no_grad():
        actions = policy(obs)
        
        # アクションスケーリング
        scaled_actions = actions * action_scale
        obs, rews, dones, infos = env.step(scaled_actions)  # スケール済みを使用
        torques = env.sim.sbody.getTorques()
        
        # データを記録
        step_data.append(cnt)
        obs_data.append(_obs_vec(obs))
        torque_data.append(torques.copy())
        
        # デバッグ表示（最初の数ステップのみ）
        if i < 3:
            print(f"Step {i}: Original action max={actions.max():.3f}, "
                  f"Scaled action max={scaled_actions.max():.3f}")
        
        if i % 20 == 0:
            print(f"Step {i+1}/{num_steps}, Total steps: {cnt}")
            print("steps:",cnt)
            print("actions :",scaled_actions)
            print("target_dof_pos:",env.target_dof_pos)
        
        cnt += 1

print(f"データ収集完了: {num_steps} steps collected with action_scale={action_scale}")

Step 0: Original action max=0.915, Scaled action max=0.915
Step 1/10, Total steps: 22
steps: 22
actions : tensor([[-0.6653, -0.4776, -0.1424,  0.9150, -0.5116, -0.1535, -1.3193,  0.0841,
          0.7004, -1.2475,  0.1423, -0.4062]], device='cuda:0')
target_dof_pos: tensor([[-0.4114, -0.3602, -1.0389,  1.9573, -1.2839, -0.1116, -1.3783, -0.0318,
         -0.3334,  0.5246, -0.7049,  0.1379]], device='cuda:0')
Step 1: Original action max=0.791, Scaled action max=0.791
Step 2: Original action max=0.704, Scaled action max=0.704
データ収集完了: 10 steps collected with action_scale=1.0


In [97]:
# for i in range(500):
#     with torch.no_grad():
#         actions = policy(obs)
#         scaled_actions = actions * action_scale
#         obs, rews, dones, infos = env.step(scaled_actions)

In [55]:
env.sim.stop()

In [57]:
env.reset()
cnt = 0

In [56]:
# 最もシンプルな保存方法
def save_simple_csv():
    if not step_data:
        print("データがありません")
        return
    
    # 基本的な辞書形式でデータを整理
    data_dict = {'step': step_data}
    
    # # Actionデータ
    # action_array = np.array(action_data)
    # for i in range(action_array.shape[1]):
    #     data_dict[f'action_{i}'] = action_array[:, i]
    
    # Observationデータ
    obs_array = np.array(obs_data)
    for i in range(obs_array.shape[1]):
        data_dict[f'obs_{i}'] = obs_array[:, i]
    
    # Torqueデータ
    torque_array = np.array(torque_data)
    for i in range(torque_array.shape[1]):
        data_dict[f'torque_{i}'] = torque_array[:, i]
    
    # DataFrameを作成して保存
    df = pd.DataFrame(data_dict)
    csv_filename = f'obs_data/cnoid_{exp_name}_ckpt{ckpt}_scale{action_scale}_rotorInertia0.1.csv'
    df.to_csv(csv_filename, index=False)
    
    print(f"シンプル版を保存: {csv_filename}")
    print(f"データ形状: {df.shape}")
    
    return df

# シンプル版を実行
df_simple = save_simple_csv()

シンプル版を保存: obs_data/cnoid_friction-walking-terrain2-kp2000kd50-kpkdrand-8_ckpt100_scale1.0_rotorInertia0.1.csv
データ形状: (313, 58)
